## Schema evolution + Delta column mapping (dev/test)

**Data contracts vs silent evolution:** `mergeSchema`/`autoMerge` let new columns flow in automatically, which is convenient but means a producer can change the shape of silver data without anyone agreeing to it - consumers downstream find out by breaking. A data contract (a versioned, explicitly agreed schema between producer and consumer, checked at write time) is the controlled alternative: evolution only happens when both sides update the contract, not silently. For this lab, `mergeSchema` is fine for learning the mechanism; a real pipeline would gate it behind a reviewed contract change.

In [0]:
dbutils.widgets.text("catalog", "dbr_dev_ua5816bd")
dbutils.widgets.text("silver_schema", "lena066636_silver")
dbutils.widgets.text("silver_table", "orders")


In [0]:
silver_table = f"{dbutils.widgets.get('catalog')}.{dbutils.widgets.get('silver_schema')}.{dbutils.widgets.get('silver_table')}"


In [0]:
from pyspark.sql import functions as F

widened_df = (spark.table(silver_table).limit(1)
    .withColumn("discount_code", F.lit(None).cast("string")))

widened_df.write.format("delta").option("mergeSchema", "true").mode("append").saveAsTable(silver_table)


### Column mapping — safe rename/drop without rewriting files

In [0]:
spark.sql(f"""
    ALTER TABLE {silver_table} SET TBLPROPERTIES (
        'delta.columnMapping.mode' = 'name',
        'delta.minReaderVersion' = '2',
        'delta.minWriterVersion' = '5'
    )
""")


In [0]:
spark.sql(f"ALTER TABLE {silver_table} RENAME COLUMN discount_code TO promo_code")
spark.sql(f"ALTER TABLE {silver_table} DROP COLUMN promo_code")
